In [ ]:
# Import required libraries

In [1]:
from pyspark.sql import SparkSession

In [2]:
from pyspark.sql.functions import *

In [3]:
# Create Spark session (FIX for Mac)
spark = SparkSession.builder \
    .appName("Movies Analysis") \
    .config("spark.hadoop.fs.defaultFS", "file:///") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/04 11:35:08 WARN Utils: Your hostname, Shridhars-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.39.120.32 instead (on interface en0)
26/05/04 11:35:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/04 11:35:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/04 11:35:10 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/05/04 11:35:10 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [ ]:
# -------------------------------
# Load dataset
# -------------------------------
df = spark.read.csv(
    "file:///Users/shridharbhat/Documents/Data Engineering/SparkSQL/movies.txt",
    header=True,
    inferSchema=True
)

In [ ]:
df.show()

In [ ]:
# -------------------------------
# Sorting
# -------------------------------
df.orderBy('Collection', ascending=False).show(5)
df.orderBy('imdb', ascending=False).show(5)

In [ ]:
# -------------------------------
# Aggregations
# -------------------------------
df.agg({'Collection':'sum'}).show()
df.agg({'Collection':'avg'}).show()
df.agg({'Collection':'avg', 'imdb':'avg'}).show()

In [ ]:
# -------------------------------
# Describe
# -------------------------------
df.describe().show()

In [ ]:
# -------------------------------
# Group By
# -------------------------------
df.groupBy('Genre').agg(max('Collection')).show()

In [ ]:
# -------------------------------
# Filtering
# -------------------------------
df.filter(df['Genre'] == 'Action').show()

In [ ]:
df.filter(
    (df['Genre'] == 'Action') & 
    (df['imdb'] > 8)
).show()

In [ ]:
df.filter(
    (~df['Genre'].isin('Action', 'Sci-Fi')) & 
    (df['imdb'] > 8)
).show()

In [ ]:
# -------------------------------
# Range filter (inclusive)
# -------------------------------
fil_res = df.filter(df['Collection'].between(500, 2000))
fil_res.show()

In [ ]:
# -------------------------------
# Add constant column
# -------------------------------
df.withColumn('Critical_hit', lit('YES')).show()

In [ ]:
# -------------------------------
# Conditional column (when)
# -------------------------------
df2 = df.withColumn(
    'Critic_hit',
    when(df['imdb'] >= 8, 'YES').otherwise('NO')
)

In [ ]:
df2.show()

In [ ]:
# -------------------------------
# Group by new column
# -------------------------------
df2.groupBy('Critic_hit').count().show()

In [ ]:
# -------------------------------
# Multi-condition when (FIXED)
# -------------------------------
x = (
    when(df['Collection'] >= 2000, 'Blockbuster')
    .when(df['Collection'] >= 1500, 'Huge Hit')
    .when(df['Collection'] >= 1000, 'Big Hit')
    .otherwise('Hit')
)

In [ ]:
result = df.withColumn('BoxOffice', x)
result.show()

In [ ]:
# -------------------------------
# UDF (FIXED LOGIC)
# -------------------------------
def box_office_verdict(coll):
    if coll >= 2000:
        return 'Blockbuster'
    elif coll >= 1500:
        return 'Huge Hit'
    elif coll >= 1000:
        return 'Big Hit'
    else:
        return 'Hit'

In [ ]:
box_office_udf = udf(box_office_verdict)

In [ ]:
result_udf = df.withColumn('BoxOffice', box_office_udf(df['Collection']))
result_udf.show()

In [ ]:
# Stop Spark
spark.stop()